# 03 — Fold-local, label-blind training

Each encoder starts from a recorded seed-matched initialization and is trained
only on its outer-training source videos. Sampling is source-uniform before a
sequence is selected, preventing long videos from dominating. Dataset folder
annotations do not enter the representation objective. Reflection augmentation
is a registered training variant, not a post-result repair.

Vanilla and reflection-augmented checkpoints use the same initialization,
source draws, target masks, geometric views, optimizer schedule, and update
count. Reflection has its own random stream, so consuming augmentation draws
cannot perturb sampling or masking. This makes their difference a paired
recipe ablation under the registered seeds—not a universal causal claim.

Checkpoint lineage binds protocol, cohort, split, allowed sources, fold, seed,
variant, and implementation. A mismatch fails instead of silently reusing a
stale model. `smoke` runs validate execution only; paper evidence requires the
locked paper profile, every registered fold and seed, and subsequent held-out
evaluation. Training cannot resolve the ethics or data-use gate.

## What experiment does this notebook perform?

This notebook performs a **paired training-recipe ablation**. An ablation is a
controlled comparison in which one ingredient is changed while the remaining
ingredients are kept matched. Here the changed ingredient is anatomical
reflection augmentation:

- `vanilla` uses the original training poses and never applies reflection;
- `reflection_augmented` reflects a sampled pose with probability 0.5 by
  reversing the horizontal coordinate and exchanging corresponding left and
  right landmarks.

The scientific motivation is to learn whether exposure to reflected examples
later improves left–right behavior. Notebook 03 cannot answer that question by
itself. Its job is to produce the correctly isolated encoders. Notebooks 04
and 05 subsequently test their held-out predictive utility and their direct
response to reflection.

The model is an **S-JEPA**, or Skeleton Joint-Embedding Predictive
Architecture. It hides some valid space-time pose patches and learns to predict
their latent representations. It is trained jointly with **VICReg**, short for
Variance–Invariance–Covariance Regularization. In simple terms, VICReg asks two
views of the same pose to have similar summaries while discouraging every
example from collapsing to the same representation and discouraging redundant
representation dimensions. The laterality target and dataset condition label
never enter either loss. “Label-blind” does not mean anatomy-blind: landmark
identities and the registered left/right mapping are still part of the pose
representation and augmentation.

Each outer fold seals away 18 or 19 source videos. A fresh encoder sees only
that fold's other 74 or 75 sources. For each fold, seeds 42 through 46 create
five reproducible optimization repeats, and each seed is matched across the two
variants. Consequently, the paper design trains $5\times5\times2=50$
encoders. These are 50 fitted models, not 50 independent datasets: folds reuse
some training sources and seeds reuse the same fold, so the job count must not
be treated as an inferential sample size.

<svg viewBox="0 0 1060 360" width="100%" role="img"
     aria-labelledby="training-flow-title training-flow-description"
     xmlns="http://www.w3.org/2000/svg">
  <title id="training-flow-title">Fold-local paired training workflow</title>
  <desc id="training-flow-description">The locked cohort is divided into five
  outer folds. For each fold, training sources feed matched vanilla and
  reflection-augmented jobs across five seeds while test sources remain sealed.
  Fifty checkpoints then move to held-out evaluation in later notebooks.</desc>
  <defs>
    <marker id="arrow03" markerWidth="8" markerHeight="8" refX="7" refY="4"
            orient="auto" markerUnits="strokeWidth">
      <path d="M0,0 L8,4 L0,8 z" fill="#475569"/>
    </marker>
  </defs>
  <style>
    .box03 { fill:#f8fafc; stroke:#334155; stroke-width:1.5; rx:10; }
    .train03 { fill:#ecfdf5; stroke:#047857; stroke-width:1.5; rx:10; }
    .hold03 { fill:#fff7ed; stroke:#c2410c; stroke-width:1.5; rx:10; }
    .variant03a { fill:#eff6ff; stroke:#2563eb; stroke-width:1.5; rx:10; }
    .variant03b { fill:#fff1f2; stroke:#e11d48; stroke-width:1.5; rx:10; }
    .line03 { stroke:#475569; stroke-width:1.8; fill:none;
              marker-end:url(#arrow03); }
    .t03 { font: 15px system-ui, sans-serif; fill:#0f172a; }
    .s03 { font: 12px system-ui, sans-serif; fill:#475569; }
    .h03 { font: 600 15px system-ui, sans-serif; fill:#0f172a; }
  </style>
  <rect class="box03" x="15" y="125" width="155" height="82"/>
  <text class="h03" x="92" y="153" text-anchor="middle">Locked cohort</text>
  <text class="s03" x="92" y="175" text-anchor="middle">93 source videos</text>
  <text class="s03" x="92" y="193" text-anchor="middle">625 sequences</text>

  <rect class="box03" x="215" y="125" width="155" height="82"/>
  <text class="h03" x="292" y="153" text-anchor="middle">Outer fold</text>
  <text class="s03" x="292" y="175" text-anchor="middle">repeat for folds 0–4</text>
  <text class="s03" x="292" y="193" text-anchor="middle">split by source video</text>
  <path class="line03" d="M170 166 L215 166"/>

  <rect class="train03" x="420" y="55" width="175" height="82"/>
  <text class="h03" x="507" y="83" text-anchor="middle">Training sources</text>
  <text class="s03" x="507" y="105" text-anchor="middle">74 or 75 sources</text>
  <text class="s03" x="507" y="123" text-anchor="middle">labels excluded from loss</text>
  <rect class="hold03" x="420" y="230" width="175" height="82"/>
  <text class="h03" x="507" y="258" text-anchor="middle">Sealed test sources</text>
  <text class="s03" x="507" y="280" text-anchor="middle">18 or 19 sources</text>
  <text class="s03" x="507" y="298" text-anchor="middle">never used in notebook 03</text>
  <path class="line03" d="M370 154 C392 154 394 96 420 96"/>
  <path class="line03" d="M370 178 C392 178 394 271 420 271"/>

  <rect class="variant03a" x="645" y="20" width="180" height="82"/>
  <text class="h03" x="735" y="48" text-anchor="middle">Vanilla</text>
  <text class="s03" x="735" y="70" text-anchor="middle">reflection probability 0</text>
  <text class="s03" x="735" y="88" text-anchor="middle">five matched seeds</text>
  <rect class="variant03b" x="645" y="125" width="180" height="82"/>
  <text class="h03" x="735" y="153" text-anchor="middle">Reflection augmented</text>
  <text class="s03" x="735" y="175" text-anchor="middle">reflection probability 0.5</text>
  <text class="s03" x="735" y="193" text-anchor="middle">five matched seeds</text>
  <path class="line03" d="M595 82 L645 61"/>
  <path class="line03" d="M595 111 L645 152"/>

  <rect class="box03" x="870" y="73" width="170" height="90"/>
  <text class="h03" x="955" y="101" text-anchor="middle">50 checkpoints</text>
  <text class="s03" x="955" y="123" text-anchor="middle">300 epochs each</text>
  <text class="s03" x="955" y="141" text-anchor="middle">1,200 updates each</text>
  <path class="line03" d="M825 61 C847 61 847 102 870 102"/>
  <path class="line03" d="M825 166 C847 166 847 133 870 133"/>

  <rect class="hold03" x="815" y="245" width="225" height="82"/>
  <text class="h03" x="927" y="273" text-anchor="middle">Held-out evaluation</text>
  <text class="s03" x="927" y="295" text-anchor="middle">not performed here</text>
  <text class="s03" x="927" y="313" text-anchor="middle">continue to notebooks 04–05</text>
  <path class="line03" d="M595 271 L815 278"/>
  <path class="line03" d="M955 163 L955 245"/>
</svg>

## How to read the live progress display

The training cell below can train many independent encoders, so it reports
progress at two levels. A **job** is one variant, outer fold, and random seed.
A variant is one registered training recipe. An outer fold identifies which
source-video group is held out, and a seed identifies one reproducible random
starting point. An **epoch** is one scheduled round of source-balanced batches,
while an optimizer update is one step that changes the learned weights.

Within a newly trained job, the display shows completed epochs and optimizer
updates, current training loss, elapsed time, and estimated time remaining. The
overall bar counts all registered jobs. A previously saved checkpoint counts as
complete only after its lineage and model-state fingerprints have been
validated.

The estimated time of arrival (ETA) is intentionally adaptive. At first it
uses the median historical duration of compatible checkpoints produced on the
same device, if any exist. Otherwise, an estimate appears after the first new
epoch. Once five new epochs are available, their recent median speed takes over.
This is more stable than extrapolating from one unusually fast or slow epoch,
but it remains an estimate: thermal throttling, other work on the computer, or
suspending the laptop can change it.

The display updates in place about once per second, so it does not create
thousands of notebook output lines. Re-running the cell safely reuses every
valid completed checkpoint. Training is checkpointed after a whole job, not
after every epoch. If execution is interrupted in the middle of a job, earlier
completed jobs remain reusable, but the interrupted job starts again.

Training loss describes the self-supervised optimization objective. It is not
validation accuracy or held-out performance. Those quantities are calculated
later, in notebooks 04 and 05, after the outer-test sources are evaluated.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display


def locate_suite_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for ancestor in (start, *start.parents):
        for candidate in (ancestor, ancestor / "neurips-laterality"):
            if (
                (candidate / "config" / "protocol.json").is_file()
                and (candidate / "laterality").is_dir()
            ):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate neurips-laterality from the current working directory."
    )


SUITE_ROOT = locate_suite_root()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))

from laterality.config import load_context

context = load_context(SUITE_ROOT / "config" / "protocol.json")
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")


def show_inline(figure):
    display(figure)
    plt.close(figure)


print(
    f"suite={SUITE_ROOT} profile={context.profile} "
    f"artifacts={context.artifact_root} protocol={context.protocol_digest[:12]}"
)

In [ ]:
import pandas as pd

from laterality.data import load_cohort
from laterality.splitting import load_splits
from laterality.training import train_selected
from laterality.visualization import training_figure
from notebook_progress import NotebookTrainingProgress

cohort = load_cohort(context)
splits = load_splits(context, cohort)
progress = NotebookTrainingProgress(refresh_seconds=1.0)
training_summaries = train_selected(
    context,
    cohort,
    splits,
    progress_callback=progress,
)
show_inline(training_figure(context, training_summaries))
pd.DataFrame(training_summaries).drop(columns="history").sort_values(
    ["variant", "fold", "seed"]
).reset_index(drop=True)

## Step-by-step interpretation of the completed paper run

### 1. First establish what completed successfully

The saved output is from the **paper profile**, not the synthetic smoke test.
It contains all 50 registered jobs: 25 vanilla checkpoints and 25
reflection-augmented checkpoints. Every row reports 1,200 optimizer updates,
which equals 300 epochs times four updates per epoch. No run stopped early and
no non-finite loss was reported. This is evidence that the fixed training plan
completed; it is not yet evidence that the representations are useful.

The repeated `enable_nested_tensor ... norm_first` messages are PyTorch
performance warnings. They say that one optional nested-tensor optimization was
not used by the Transformer configuration. They do not indicate missing data,
leakage, an invalid checkpoint, or failed optimization. A traceback, a
non-finite-loss exception, a lineage mismatch, or a missing row would be a
substantively different warning sign.

### 2. Read the left plot as an optimization diagnostic

The solid line is the mean training loss at each epoch across the 25 fold/seed
jobs for that variant. The translucent band runs from the smallest to the
largest observed job loss at that epoch. It is a **range**, not a confidence
interval, standard error, or test of a difference between variants.

In the saved run, mean loss changed as follows:

| Epoch | Vanilla mean loss | Reflection-augmented mean loss |
|---:|---:|---:|
| 1 | 11.923 | 11.870 |
| 10 | 3.015 | 3.060 |
| 50 | 1.829 | 1.902 |
| 100 | 1.535 | 1.605 |
| 200 | 1.309 | 1.350 |
| 300 | 1.241 | 1.299 |

Both variants therefore reduced their mean objective by about 89% from epoch 1
to epoch 300, and all 50 individual jobs ended below their own epoch-1 loss.
Most of the reduction happened early. Improvement continued after epoch 200,
but it was smaller: approximately 0.069 for vanilla and 0.052 for the augmented
recipe. The average over the last 20 epochs was 1.246 for vanilla and 1.299 for
reflection augmentation, close to the final-epoch means. This pattern is
consistent with stable optimization approaching a plateau rather than obvious
divergence.

Individual epoch losses still fluctuate because every epoch samples sequences,
masks, and geometric views. The checkpoint is deliberately the fixed epoch-300
checkpoint, not whichever epoch happened to have the smallest training loss.
The median minimum-loss epoch was 280 for vanilla and 276 for reflection
augmentation, which also shows why a single lowest point should not be treated
as a selected model.

### 3. Do not use the loss gap to choose a winning variant

Final loss averaged 1.241 for vanilla and 1.299 for reflection augmentation.
In matched fold/seed pairs, augmented minus vanilla final loss averaged +0.058,
had a median of +0.034, and ranged from -0.076 to +0.228. The augmented job had
the smaller final loss in 6 of 25 pairs.

These are useful optimization descriptions, but they do **not** demonstrate
that reflection augmentation harms or helps the scientific outcome. The
augmented model is trained on a different input distribution because each
sampled pose has a 0.5 chance of being reflected. Moreover, this self-supervised
objective is not held-out laterality error. For example, fold 2/seed 44 ended at 1.175 for
vanilla and 1.171 for augmentation—almost identical—whereas fold 4/seed 45
ended at 1.168 and 1.396. Neither pair tells us which encoder predicts the
coordinate-derived target better or transforms more correctly under reflection.
Those comparisons require the paired held-out measurements in notebooks 04
and 05.

Loss is also not a percentage. A loss of 1.2 does not mean 120% error or 80%
accuracy. It is the sum of the latent prediction objective and a weighted
VICReg regularizer. Its scale is meaningful mainly for detecting learning,
instability, and unusual differences under the fixed implementation.

### 4. Read the right plot as a fairness-of-exposure check

For each job, the right panel divides the most frequently drawn training source
by the least frequently drawn training source. Exactly 1.0 would mean identical
counts. The observed ratios ranged from 1.054 to 1.087, so the most frequent
source received at most about 8.7% more draws than the least frequent source.

Every job made 24,000 source draws: 300 epochs times four batches times 20
sequences. Each epoch first visits every one of its 74 or 75 training sources,
guaranteeing at least 300 draws per source, and then uses source-uniform padding
to reach the fixed four-batch budget. Across jobs, the recorded extremes were
309 to 340 draws. This small spread is compatible with the intended balanced
random padding and gives no sign that a long source video dominated merely
because it contained more extracted sequences.

The vanilla and augmented job in every matched fold/seed pair have identical
minimum and maximum draw counts. This is expected because their source-sampling
stream is shared, while reflection decisions use a separate stream. The two
variant columns therefore have the same vertical pattern. Within either column,
multiple jobs with the same ratio can overlap, so fewer than 25 dots may be
visually distinguishable.

### 5. Interpret each table column literally

- `variant` identifies the training recipe, not a result category.
- `fold` identifies which source-video group was sealed away. Fold 0 is not
  earlier, easier, or more important than fold 4.
- `seed` identifies a reproducible random initialization and random stream. It
  is not a subject identifier and larger seed numbers are not better.
- `optimizer_updates` verifies equal compute. All rows should show 1,200.
- `final_loss` is the mean total training loss in epoch 300. It is neither a
  validation loss nor test performance.
- `minimum_source_draws` and `maximum_source_draws` summarize source exposure.
  Their ratio is what appears in the right plot.
- `checkpoint` is the local artifact used downstream. Its lineage records the
  allowed training sources and forbidden test sources so an incompatible file
  fails validation instead of being silently reused.

### 6. State the narrow conclusion

Notebook 03 supports the following conclusion: **all registered fold-local,
label-blind training jobs completed under equal update budgets; both recipes
learned a substantially lower and stable training objective; and source
exposure remained closely balanced and paired across recipes.**

It does not yet support “reflection augmentation is better,” “the encoder is
reflection-equivariant,” “the target is predictable on held-out sources,” or
any diagnostic or clinical claim. Notebook 04 next applies each fold-local
checkpoint to the sources that checkpoint never saw and fits read-outs without
opening the outer-test targets. Notebook 05 aggregates those paired held-out
results and applies the registered decision rules. Only those stages can answer
the scientific comparison posed here.